<a href="https://colab.research.google.com/github/zackdihel/ECON-5200-Data-Analytics/blob/main/Lab%2012/%5BLab_12%5D_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse

In [3]:
url = "https://raw.githubusercontent.com/zackdihel/ECON-5200-Data-Analytics/refs/heads/main/Lab%2012/Zillow_ZHVI_2026_Micro.csv"
df = pd.read_csv(url)

df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [4]:
formula = 'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [5]:
model = smf.ols(formula, data=df)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        17:56:42   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [6]:
y_pred = results.predict(df)

In [7]:
model_rmse = rmse(df['Home_Value'], y_pred)

print(f"\nThe predictive RMSE is: ${model_rmse:,.2f}")


The predictive RMSE is: $42,316.69


**AI Expansion**

In [8]:
"""
=============================================================================
  HEDONIC PRICING MODEL — INTERACTIVE RESIDUAL FORENSICS DASHBOARD
  Built with statsmodels OLS + Plotly Express
=============================================================================
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go
from sklearn.datasets import fetch_california_housing  # swap for your own data


# ── 0. DATA  ─────────────────────────────────────────────────────────────────
# Replace this block with your own DataFrame + formula if you already have a
# fitted statsmodels results object — just skip ahead to Section 2.

housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Quick feature log-transform (common in hedonic models to linearise prices)
df["log_price"]       = np.log(df["MedHouseVal"])
df["log_income"]      = np.log(df["MedInc"])
df["log_ave_rooms"]   = np.log(df["AveRooms"].clip(lower=0.01))
df["log_population"]  = np.log(df["Population"].clip(lower=1))


# ── 1. FIT THE OLS MODEL  ─────────────────────────────────────────────────────
feature_cols = [
    "log_income", "log_ave_rooms", "log_population",
    "HouseAge", "Latitude", "Longitude"
]

X = sm.add_constant(df[feature_cols])   # statsmodels needs the intercept added manually
y = df["log_price"]

model   = sm.OLS(y, X)                  # define the OLS specification
results = model.fit()                   # .fit() returns a RegressionResultsWrapper

print(results.summary())               # inspect coefficients, R², F-stat, etc.


# ── 2. EXTRACT RESIDUALS & FITTED VALUES FROM THE RESULTS OBJECT  ─────────────
# `results.fittedvalues`  → pandas Series of ŷ  (X @ β̂)
# `results.resid`         → pandas Series of ε̂  (y − ŷ)
# Both are aligned to the original index, so we can column-bind safely.

fitted    = results.fittedvalues          # predicted values on the log scale
residuals = results.resid                 # raw OLS residuals  ε̂ = y − ŷ

# RMSE (replicate your lab metric)
rmse = np.sqrt(np.mean(residuals ** 2))
print(f"\nRMSE: {rmse:.4f}")


# ── 3. BUILD THE FORENSICS DATAFRAME  ────────────────────────────────────────
# Attach house-level metadata so we can hover-inspect individual observations.

forensics = pd.DataFrame({
    "fitted"    : fitted,
    "residual"  : residuals,
    "actual"    : y,
    "log_income": df["log_income"],
    "latitude"  : df["Latitude"],
    "longitude" : df["Longitude"],
    "house_age" : df["HouseAge"],
}, index=df.index)

# ── 3a. FLAG OUTLIERS  ───────────────────────────────────────────────────────
# Outliers are observations whose residual lies more than ±2 std-devs from 0.
# Flagging is done BEFORE visualisation so the column drives Plotly's color map.

std_resid     = residuals.std()                       # σ̂ of residuals
upper_fence   = +2 * std_resid
lower_fence   = -2 * std_resid

forensics["outlier"] = np.where(
    (forensics["residual"] > upper_fence) |
    (forensics["residual"] < lower_fence),
    "Outlier (|ε| > 2σ)",                             # label for legend
    "Normal"
)

# Proportion of flagged observations
n_outliers = (forensics["outlier"] == "Outlier (|ε| > 2σ)").sum()
pct        = 100 * n_outliers / len(forensics)
print(f"Outliers flagged: {n_outliers} ({pct:.1f}%)")


# ── 4. BUILD THE INTERACTIVE SCATTER PLOT  ───────────────────────────────────
# Color map: soft steel-blue for inliers, stark crimson for outliers.

COLOR_MAP = {
    "Normal"              : "#5B8DB8",   # muted steel blue
    "Outlier (|ε| > 2σ)"  : "#DC143C",   # CSS Crimson
}

fig = px.scatter(
    forensics,
    x          = "fitted",
    y          = "residual",
    color      = "outlier",                         # drives the two-tone palette
    color_discrete_map = COLOR_MAP,
    opacity    = 0.65,
    hover_data = {                                  # rich tooltip on hover
        "fitted"    : ":.4f",
        "residual"  : ":.4f",
        "actual"    : ":.4f",
        "log_income": ":.3f",
        "house_age" : True,
        "latitude"  : ":.3f",
        "longitude" : ":.3f",
    },
    title      = (
        f"Residual Forensics Dashboard — Hedonic Pricing OLS<br>"
        f"<sup>RMSE = {rmse:.4f} | σ̂(ε) = {std_resid:.4f} | "
        f"Outliers = {n_outliers} ({pct:.1f}%)</sup>"
    ),
    labels     = {
        "fitted"   : "Fitted Values  ŷ  (log scale)",
        "residual" : "Residuals  ε̂ = y − ŷ",
        "outlier"  : "Observation type",
    },
    template   = "plotly_dark",
)


# ── 5. ZERO-LINE & FENCE ANNOTATIONS  ────────────────────────────────────────
# add_hline uses graph_objects under the hood; no deprecated functions are used.

# Zero-line (ideal residual mean)
fig.add_hline(
    y           = 0,
    line_color  = "#FFFFFF",
    line_width  = 1.6,
    line_dash   = "solid",
    annotation_text      = "ε̂ = 0",
    annotation_position  = "bottom right",
    annotation_font_color= "#FFFFFF",
)

# ±2σ fences
for fence, label in [(upper_fence, "+2σ"), (lower_fence, "−2σ")]:
    fig.add_hline(
        y           = fence,
        line_color  = "#DC143C",
        line_width  = 1.2,
        line_dash   = "dash",
        annotation_text       = label,
        annotation_position   = "bottom right",
        annotation_font_color = "#DC143C",
    )


# ── 6. STYLE REFINEMENTS  ────────────────────────────────────────────────────
fig.update_traces(
    marker=dict(size=5, line=dict(width=0.4, color="rgba(255,255,255,0.3)"))
)

fig.update_layout(
    font_family   = "IBM Plex Mono, monospace",
    title_font    = dict(size=15),
    width         = 1000,
    height        = 620,
    legend        = dict(title="", orientation="h", yanchor="bottom",
                         y=1.02, xanchor="right", x=1),
    xaxis         = dict(showgrid=True, gridcolor="rgba(255,255,255,0.08)"),
    yaxis         = dict(showgrid=True, gridcolor="rgba(255,255,255,0.08)",
                         zeroline=False),   # our manual hline replaces the default zeroline
    plot_bgcolor  = "#0D1117",
    paper_bgcolor = "#0D1117",
)


# ── 7. RENDER  ───────────────────────────────────────────────────────────────
fig.show()                              # opens in default browser
# fig.write_html("residual_forensics.html")   # uncomment to save as standalone file

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.619
Model:                            OLS   Adj. R-squared:                  0.619
Method:                 Least Squares   F-statistic:                     5580.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        17:57:29   Log-Likelihood:                -7702.1
No. Observations:               20640   AIC:                         1.542e+04
Df Residuals:                   20633   BIC:                         1.547e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const            -22.7510      0.333    -68.